In [1]:
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

True

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    ChatPromptTemplate
)

llm = ChatOpenAI(temperature=0, model_name ="gpt-4o-mini")

In [3]:
llm.invoke("What is ai?").content

'Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. It encompasses a variety of technologies and methodologies that enable computers to perform tasks that typically require human intelligence. These tasks include problem-solving, understanding natural language, recognizing patterns, making decisions, and learning from experience.\n\nAI can be categorized into several types:\n\n1. **Narrow AI**: Also known as weak AI, this type is designed to perform a specific task, such as voice recognition, image analysis, or playing chess. Most AI applications today fall into this category.\n\n2. **General AI**: Also known as strong AI, this type would have the ability to understand, learn, and apply intelligence across a wide range of tasks, similar to a human. General AI remains largely theoretical and has not yet been achieved.\n\n3. **Machine Learning (ML)**: A subset of AI that involves training algorithms 

In [5]:
from langchain_core.output_parsers import StrOutputParser

prompt ="""Given the user review below, classify it as either being about "Positive" or "negative".
DOn not respond with more than one word.

Review : {review}
classification:"""

template = ChatPromptTemplate.from_template(prompt)

sentimentChain = template |llm | StrOutputParser()

#eview = "Thank you so much for providing such a great plateform for learning. I am really happy with the service."
review = "iam not happy with the service. Its not good"
sentimentChain.invoke({'review': review})

'Negative'

In [6]:
positive_prompt = """
                You are expert in writing reply for positive reviews.
                You need to encourage the user to share their experience on social media.
                Review : {review}
                Answer:"""

positive_template = ChatPromptTemplate.from_template(positive_prompt)
positive_chain =  positive_template | llm | StrOutputParser()

In [7]:
negative_prompt = """
                You are expert in writing reply for negative reviews.
                You need first to apologize for the inconvenience caused to the user.
                You need to encourage the user to share their concern on following email:'adirajuadithya@company.com'
                Review:{review}
                Answer:
                """

negative_template = ChatPromptTemplate.from_template(negative_prompt)
negative_chain = negative_template | llm | StrOutputParser()


In [8]:
def decisionReviewer(info):
    if 'positive' in info['sentiment'].lower():
        return positive_chain
    else:
        return negative_chain

In [9]:
sentimentChain = template | llm | StrOutputParser()
sentimentChain.invoke({"review" : "I am not happy with the service. It is not good."})

'Negative'

In [10]:
from langchain_core.runnables import RunnableLambda

full_chain = {"sentiment": sentimentChain, 'review': lambda x: x['review']} | RunnableLambda(decisionReviewer)

#{"sentiment":"Negative", "review":"I am not happy with the service. It is not good."} ---> LAMBDA

full_chain


{
  sentiment: ChatPromptTemplate(input_variables=['review'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['review'], input_types={}, partial_variables={}, template='Given the user review below, classify it as either being about "Positive" or "negative".\nDOn not respond with more than one word.\n\nReview : {review}\nclassification:'), additional_kwargs={})])
             | ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001FAFDA89F50>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001FAFDF0BBD0>, root_client=<openai.OpenAI object at 0x000001FAFD89F990>, root_async_client=<openai.AsyncOpenAI object at 0x000001FAFDC34810>, model_name='gpt-4o-mini', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'))
             | StrOutputParser(),
  review: RunnableLambda(lambda x: x['review'])
}
| RunnableLambda(decisionRe

In [11]:
review = "Thank you so much for providing such a great plateform for learning. I am really happy with the service."
#review = "I am not happy with the service. It is not good."

output = full_chain.invoke({'review': review})
print(output)

Thank you so much for your kind words! We're thrilled to hear that you're enjoying our platform and finding it valuable for your learning journey. Your satisfaction means the world to us! If you’d like to share your experience with others, we’d love for you to post about it on social media. Your support helps us grow and reach more learners like you. Thank you again for being a part of our community!


### Make Custom Chain Runnables with RunnablePassthrough and RunnableLambda
- This is useful for formatting or when you need functionality not provided by other LangChain components, and custom functions used as Runnables are called RunnableLambdas.



In [12]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

def char_counts(text):
    return len(text)

def word_counts(text):
    return len(text.split())

def pricing(text):
    return len(text.split()) * 0.01

prompt = ChatPromptTemplate.from_template("Explain these inputs in 5 sentences : {input1} and {input2}")

prompt

ChatPromptTemplate(input_variables=['input1', 'input2'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input1', 'input2'], input_types={}, partial_variables={}, template='Explain these inputs in 5 sentences : {input1} and {input2}'), additional_kwargs={})])

In [13]:
chain = prompt | llm | StrOutputParser()

output = chain.invoke({'input1': 'Earth is planet', 'input2' : 'Sun is a star'})

print(output)


Earth is the third planet from the Sun in our solar system and is the only known celestial body to support life. It has a diverse range of environments, including oceans, mountains, and forests, which contribute to its rich biodiversity. The Sun, on the other hand, is a massive ball of gas primarily composed of hydrogen and helium, and it serves as the central star of our solar system. It provides the necessary light and heat that sustain life on Earth and drives various natural processes, such as photosynthesis. Together, Earth and the Sun are part of a complex gravitational system that influences the planet's climate and seasons.


In [14]:
# f(x) = x
# def passthrough(text):
#.   return str(text)
chain = prompt | llm | StrOutputParser() | {'char_counts': RunnableLambda(char_counts), 
                                            'word_counts': RunnableLambda(word_counts), 
                                            'cost': RunnableLambda(pricing),
                                            'output': RunnablePassthrough()} #RunnableLambda(passthrough)

output = chain.invoke({'input1': 'Earth is planet', 'input2': 'Sun is star'})

In [15]:
print(output)

{'char_counts': 622, 'word_counts': 105, 'cost': 1.05, 'output': "Earth is the third planet from the Sun in our solar system and is unique for its ability to support life. It has a diverse range of ecosystems, climates, and geological features, making it a dynamic and complex environment. The Sun, on the other hand, is a massive star at the center of our solar system, providing the necessary light and heat for life on Earth. It is primarily composed of hydrogen and helium and undergoes nuclear fusion, which generates energy. Together, the Earth and Sun interact in a gravitational relationship that influences the planet's climate, weather patterns, and the cycles of day and night."}


### Custom Chain using `@chain` decorator

Custom Chains with @chain Decorator in LangChain (LCEL, v0.3.x)

1. What is the `@chain` Decorator?

- The @chain decorator is a feature of LangChain’s LCEL API (Expression Language).
- It transforms any standard Python function into a Runnable Chain.
- This allows you to wrap arbitrary Python logic—including loops, conditionals, and composition of other chains—into a component that behaves just like any other chain in LCEL (i.e., it supports .invoke(), .batch(), etc.).

2. Why Use `@chain`?

For complex, custom workflows that can’t be easily expressed by chaining with the | operator.
To combine multiple sub-chains, add custom logic, or perform steps that need Python code (like aggregation, formatting, conditional logic).
It enables full flexibility within the LCEL framework: your custom function becomes a first-class Runnable.

3. Example Explained


4. How Does it Work?

The decorator wraps your function into a Runnable-compatible object.
Supports all LCEL operations (invoke, batch, etc.).
You can include arbitrary Python code—loops, conditionals, error handling, etc.
The function input is a single parameter (often a dict).
The output can be any object—dict, list, string, etc.




In [16]:
system = SystemMessagePromptTemplate.from_template('You are {school} teacher. You answer in short sentences.')

question = HumanMessagePromptTemplate.from_template('tell me about the {topics} in {points} points')

messages = [system,question]
template = ChatPromptTemplate(messages)
fact_chain = template | llm | StrOutputParser()

output = fact_chain.invoke({'school': 'primary', 'topics': 'solar system', 'points': 2})
print(output)

1. The solar system consists of the Sun and eight planets, including Earth.  
2. It also contains moons, asteroids, comets, and dwarf planets like Pluto.


In [17]:
question = HumanMessagePromptTemplate.from_template('write poem on {topics} in {sentences} lines')

messages = [system,question]
template = ChatPromptTemplate(messages)
poem_chain = template | llm | StrOutputParser()

output = poem_chain.invoke({'school':'primary', 'topics': 'solar system', 'sentences': 2})
print(output)

Planets dance in a cosmic swirl,  
Stars shine bright in the vast, dark world.


In [19]:
from langchain_core.runnables import chain

@chain
def custom_chain(par):
    return {
        'fact': fact_chain.invoke(par),
        'poem': poem_chain.invoke(par),
    }


4. How Does it Work?

- The decorator wraps your function into a Runnable-compatible object.
- Supports all LCEL operations (invoke, batch, etc.).
- You can include arbitrary Python code—loops, conditionals, error handling, etc.
- The function input is a single parameter (often a dict).
- The output can be any object—dict, list, string, etc.

In [21]:
par = {'school': 'primary', 'topics': 'solar system', 'points': 2, 'sentences':2}
output =  custom_chain.invoke(par)

print(output['fact'])
print('\n\n')
print(output['poem'])

1. The solar system consists of the Sun and eight planets, including Earth.  
2. It also contains moons, asteroids, comets, and dwarf planets like Pluto.



Planets dance in a cosmic swirl,  
Stars shine bright in the vast, dark world.
